In [1]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import modAL

df = pd.read_csv('ourall.csv')
# for i in df.columns:
#     if df[i].dtype is not np.float64:
#         df[i] = df[i].astype(np.float64)
print(df.columns)
df = df[df['B4TOD4'] != 0.005]
df = df[df['OMEGA5'] != 15]
df = df[df['OMEGA5'] != -15]
# scaler = MinMaxScaler()
# Fit and transform the data
# df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
# train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
scaler = MinMaxScaler()
# Fit and transform the data
df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
X_pool = df.drop("PT_LOSS", axis=1).values
y_pool = df["PT_LOSS"].values.reshape(-1, 1)
print(df['B4TOD4'].unique(),'B4TOD4')
print(df['B5TOB4'].unique(),'B5TOB4')
print(df['R45TOB4'].unique(),'R45TOB4')
print(df['OMEGA5'].unique(),'OMEGA5')
print(df['GAMMAU'].unique(),'GAMMAU')
print(df['ALPHA4'].unique(),'ALPHA4')
print(df['RE'].unique(),'RE')

print(len(y_pool))
print(X_pool.shape, y_pool.shape)
# print(X_pool,type(X_pool[100]))
# print(X_pool)
# tX, ty = torch.from_numpy(X_pool).float(), torch.from_numpy(y_pool).float()
# dataset = create_dataset_

Index(['B4TOD4', 'B5TOB4', 'R45TOB4', 'OMEGA5', 'GAMMAU', 'ALPHA4', 'RE',
       'PT_LOSS'],
      dtype='object')
[0.         0.42857143 0.71428571 1.        ] B4TOD4
[0.5 1.  0. ] B5TOB4
[0.        0.3902439 1.       ] R45TOB4
[0.  0.5 1. ] OMEGA5
[0.  1.  0.5] GAMMAU
[0.  0.2 0.4 0.6 0.8 1. ] ALPHA4
[0. 1.] RE
3888
(3888, 7) (3888, 1)


In [2]:
from typing import Optional  # Добавлено
import torch
import gpytorch
import numpy as np
from gpytorch.models import ExactGP
from gpytorch.means import ConstantMean
from gpytorch.kernels import RBFKernel, MaternKernel, ScaleKernel, PeriodicKernel  # Добавлены ScaleKernel и PeriodicKernel
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.constraints import Interval
from sklearn.base import BaseEstimator  # Добавлено
from modAL.models import ActiveLearner  # Добавлено, если используется modAL
from sklearn.metrics import mean_absolute_error,r2_score

class GPModel(ExactGP):
    def __init__(
        self,
        train_x: Optional[torch.Tensor] = None,
        train_y: Optional[torch.Tensor] = None,
        likelihood: Optional[GaussianLikelihood] = None,
        kernel_type: str = 'matern',
        nu: float = 0.5,
        lengthscale_constraint: Optional[Interval] = None,
        outputscale_constraint: Optional[Interval] = None
    ):
        """
        Улучшенная GP модель с поддержкой разных ядер и modAL
        
        Параметры:
        ----------
        kernel_type : str
            Тип ядра ('matern', 'rbf', 'periodic')
        nu : float
            Параметр гладкости для ядра Matern
        """
        if likelihood is None:
            likelihood = GaussianLikelihood()
        
        super().__init__(train_x, train_y, likelihood)
        
        # Средняя функция
        self.mean_module = ConstantMean()
        
        # Инициализация ограничений
        if lengthscale_constraint is None:
            lengthscale_constraint = Interval(1e-4, 1e4)  # Изменено на Interval
        if outputscale_constraint is None:
            outputscale_constraint = Interval(1e-4, 1e4)  # Изменено на Interval
        
        # Выбор ядра
        self.kernel_type = kernel_type.lower()
        if self.kernel_type == 'matern':
            base_kernel = MaternKernel(
                nu=nu,
                lengthscale_constraint=lengthscale_constraint
            )
        elif self.kernel_type == 'rbf':
            base_kernel = RBFKernel(
                lengthscale_constraint=lengthscale_constraint
            )
        elif self.kernel_type == 'periodic':
            base_kernel = PeriodicKernel(
                lengthscale_constraint=lengthscale_constraint
            )
        else:
            raise ValueError(f"Unknown kernel type: {kernel_type}. Supported: 'matern', 'rbf', 'periodic'")
        
        self.covar_module = ScaleKernel(
            base_kernel,
            outputscale_constraint=outputscale_constraint
        )
        
        self.initialize_parameters()
    
    def initialize_parameters(self):
        """Инициализация параметров модели"""
        if self.train_inputs is not None:
            init_lengthscale = self.train_inputs[0].std().item()
            self.covar_module.base_kernel.lengthscale = init_lengthscale
    
    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)
    
    def fit(self, train_x, train_y, n_iters=80, lr=0.1, verbose=True):
        # Преобразуем данные в тензоры, если это ещё не сделано
        if not isinstance(train_x, torch.Tensor):
            train_x = torch.tensor(train_x, dtype=torch.float32)
        if not isinstance(train_y, torch.Tensor):
            train_y = torch.tensor(train_y, dtype=torch.float32)
    
        # Убедимся, что y — одномерный тензор (иначе squeeze)
        if len(train_y.shape) > 1:
            train_y = train_y.squeeze(-1)
    
        self.set_train_data(train_x, train_y, strict=False)
        self.train()
        self.likelihood.train()
    
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        mll = gpytorch.mlls.ExactMarginalLogLikelihood(self.likelihood, self)
    
        for i in range(n_iters):
            optimizer.zero_grad()
            output = self(train_x)
            loss = -mll(output, train_y)
    
            # Если loss не скаляр (например, из-за batched данных), берём сумму
            if loss.dim() > 0:
                loss = loss.sum()
    
            loss.backward()  # Теперь градиенты вычислятся корректно
            optimizer.step()
    
            if verbose and (i % 10 == 0 or i == n_iters - 1):
                print(f'Iter {i+1}/{n_iters} - Loss: {loss.item():.3f}')
            
    def predict(self, test_x, return_std=True):
        """
        Предсказание (совместимо с modAL)
        """
        if not isinstance(test_x, torch.Tensor):
            test_x = torch.tensor(test_x, dtype=torch.float32)
            
        self.eval()
        self.likelihood.eval()
        
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            observed_pred = self.likelihood(self(test_x))
            if return_std:
                return observed_pred.mean.numpy(), observed_pred.stddev.numpy()
            return observed_pred.mean.numpy()
    
    def score(self, X, y):
        """
        Метод для совместимости с modAL (возвращает negative MSE)
        """
        pred, _ = self.predict(X)
        return -np.mean((pred - y)**2).item()


class GPModelSklearnWrapper(BaseEstimator):
    """Обертка для совместимости с sklearn/modAL"""
    def __init__(self, kernel_type='matern', nu=0.5, n_iters=100, lr=0.1):
        self.kernel_type = kernel_type
        self.nu = nu
        self.n_iters = n_iters
        self.lr = lr
        self.model = None
        
    def fit(self, X, y):
        self.model = GPModel(
            kernel_type=self.kernel_type,
            nu=self.nu
        )
        self.model.fit(X, y, n_iters=self.n_iters, lr=self.lr)
        return self
    
    def predict(self, X, return_std=False):
        if self.model is None:
            raise RuntimeError("Model not trained yet")
        return self.model.predict(X, return_std=return_std)
    
    def score(self, X, y):
        return self.model.score(X, y)


# Пример использования:
if __name__ == "__main__":
    # 1. Генерация данных
    np.random.seed(42)
    # X_pool = np.random.rand(100, 1)
    # y_pool = np.sin(X_pool * 2 * np.pi).ravel() + np.random.normal(0, 0.1, size=100)
    
    # 2. Подготовка данных
    train_idx = np.random.choice(range(X_pool.shape[0]), size=10, replace=False)
    train_x = X_pool[train_idx]
    train_y = y_pool[train_idx]
    
    test_idx = np.random.choice(range(X_pool.shape[0]), size=1, replace=False)
    test_x = X_pool[test_idx]
    test_y = y_pool[test_idx]
    # 3. Обучение модели
    model = GPModelSklearnWrapper(kernel_type='matern', nu=0.5)
    model.fit(train_x, train_y)
    
    # 4. Предсказание
    pred_mean, pred_std = model.predict(test_x, return_std=True)
    # print('mae',mean_absolute_error(y_test,pred_mean))
    # print('r2',r2_score(y_test,pred_mean))
    print("Predictions:", pred_mean)
    print("Uncertainty:", pred_std)

Iter 1/100 - Loss: 1.777
Iter 11/100 - Loss: 1.340
Iter 21/100 - Loss: 0.877
Iter 31/100 - Loss: 0.429
Iter 41/100 - Loss: 0.108
Iter 51/100 - Loss: 0.035
Iter 61/100 - Loss: 0.049
Iter 71/100 - Loss: 0.029
Iter 81/100 - Loss: 0.029
Iter 91/100 - Loss: 0.025
Iter 100/100 - Loss: 0.023
Predictions: [0.23327386]
Uncertainty: [0.14606106]


In [3]:
def NA_QBC(committee, X_sample):
    grads = []
    for i in range(X_sample.shape[0]):
        x_s = np.array([X_sample[i]])
        qbc_loss, tx = qbc(committee, x_s)
        if tx.grad is not None:
            tx.grad.zero_()
        qbc_loss.backward(retain_graph=True)
        grads.append(tx.grad.detach().clone())
    return grads

def get_steps(x):
    steps = [{-1:None, 0: 0 , 1: None} for i in range(7)]
# [0.01 0.04 0.06 0.08] B4TOD4
# [1.25 1.5  1.  ] B5TOB4
# [0.9 2.5 5. ] R45TOB4
# [-5  0  5] OMEGA5
# [-0.5  0.5  0. ] GAMMAU
# [10 20 30 40 50 60] ALPHA4
# [100000 500000] RE
    
# [0.         0.42857143 0.71428571 1.        ] B4TOD4
# [0.5 1.  0. ] B5TOB4
# [0.        0.3902439 1.       ] R45TOB4
# [0.  0.5 1. ] OMEGA5
# [0.  1.  0.5] GAMMAU
# [0.  0.2 0.4 0.6 0.8 1. ] ALPHA4
# [0. 1.] RE

    
    if x[0] <= 0.2:
        steps[0][1] = 0.42857143
        steps[0][-1] = 0
    elif x[0] <= 0.5:
        steps[0][1] = 0.28571428
        steps[0][-1] = -0.42857143
    elif x[0] <= 0.75:
        steps[0][1] = 0.28571429
        steps[0][-1] = -0.28571428
    elif x[0] > 0.72:
        steps[0][1] = 0
        steps[0][-1] = -0.28571429

    if x[1] <= 0.15:
        steps[1][1] = 0.5
        steps[1][-1] = 0
    elif x[1] <= 0.6:
        steps[1][1] = 0.5
        steps[1][-1] = -0.5
    elif x[1] > 0.7:
        steps[1][1] = 0
        steps[1][-1] = -0.5

    if x[2] <= 0.35:
        steps[2][1] = 0.3902439
        steps[2][-1] = 0
    elif x[2] <= 0.7:
        steps[2][1] = 0.6097561
        steps[2][-1] = -0.3902439
    elif x[2] > 0.7:
        steps[2][1] = 0
        steps[2][-1] = -0.6097561

    if x[3] <= 0.15:
        steps[3][1] = 0.5
        steps[3][-1] = 0
    elif x[3] <= 0.6:
        steps[3][1] = 0.5
        steps[3][-1] = -0.5
    elif x[3] > 0.7:
        steps[3][1] = 0
        steps[3][-1] = -0.5

    if x[4] <= 0.25:
        steps[4][1] = 0.5
        steps[4][-1] = 0
    elif x[4] <= 0.75:
        steps[4][1] = 0.5
        steps[4][-1] = -0.5
    elif x[4] > 0.7:
        steps[4][1] = 0
        steps[4][-1] = -0.5

    if x[5] <= 0.15:
        steps[5][1] = 0.2
        steps[5][-1] = 0
    elif x[5] <= 0.25:
        steps[5][1] = 0.2
        steps[5][-1] = -0.2
    elif x[5] <= 0.55:
        steps[5][1] = 0.2
        steps[5][-1] = -0.2
    elif x[5] <= 0.75:
        steps[5][1] = 0.2
        steps[5][-1] = -0.2
    elif x[5] <= 0.87:
        steps[5][1] = 0.2
        steps[5][-1] = -0.2
    elif x[5] > 0.95:
        steps[5][1] = 0
        steps[5][-1] = -0.2

    if x[6] <= 0.5:
        steps[6][1] = 1.
        steps[6][-1] = 0
    elif x[6] >= 0.6:
        steps[6][1] = 0.0
        steps[6][-1] = -1.
    return steps
        


# [0.005 0.02  0.1  ] B4TOD4
# [1.25 1.   1.5 ] B5TOB4
# [0.9 5.  2.5] R45TOB4
# [-5.  5.  0.] OMEGA5
# [-0.5  0.5  0. ] GAMMAU
# [10. 30. 60. 20. 40. 50.] ALPHA4
# [100000. 500000.] RE

def Lbnd(X_sample, min_max_val =[[0.,1.],[0.,1.],[0.,1.],[0.,1.],[0.,1.],[0.,1.],[0.,1.]]):
    
    l_grad_bnd = []
    for i in range(X_sample.shape[0]):
        l_grad = []
        x_s = np.array([X_sample[i]])
        for ind in range(len(x_s[0])):
            if  x_s[0][ind] >= min_max_val[ind][1]:
                l_grad.append(-1.0)
            elif min_max_val[ind][0] < x_s[0][ind] < min_max_val[ind][1]:
                l_grad.append(0.0)
            elif  x_s[0][ind] <= min_max_val[ind][0]:
                l_grad.append(1.0)
        l_grad_bnd.append(torch.tensor([l_grad]))
    return l_grad_bnd
    
# def NA_query_strategy(comittee, X_sample):
#     for i in range(4):
#         grads = qbc(comittee, X_sample) # (N, M) -> N
#         l_grad_bnd = Lbnd(X_sample)
#         result = [a - b for a, b in zip(grads, l_grad_bnd)]
#         print(result, 'result grads - l_bnd')
#         sign = [torch.where(tensor > 0.00001, torch.tensor(1), 
#                   torch.where(tensor < 0, torch.tensor(-1), torch.tensor(0))) for tensor in result]
#         # sign = [torch.where(tensor > 0, torch.tensor(1)), 
#         #   torch.where(tensor < 0, torch.tensor(-1)), torch.where(tensor <= 0.00001, torch.tensor(0)) for tensor in result]
#         print(sign, 'sign grads - l_bnd')
#         for ind in range(len(sign)):
#             real_steps = []
#             sign_list = sign[ind].squeeze().tolist()
#             steps = get_steps(X_sample[ind])
#             x_gen = None
#             for index, values in enumerate(sign_list):
#                 real_steps.append(steps[index][values])
#             print(real_steps,'real_steps')
#             x_gen = X_sample[ind] + real_steps
#             X_sample[ind] = x_gen.copy()
#         print(X_sample,"X_sample",i)
#     return None, X_sample


def get_new_y(X_sampels,X_pool,y_pool):
    index_list = []
    X_study = []
    y_study = []
    for target_row in X_sampels:
        # print(X_sampels)
        print(target_row)
        # index = np.where((X_pool == target_row).all(axis=1))[0]
        index = np.where(np.all(np.isclose(X_pool, target_row), axis=1))[0]
        # index = np.where(np.all(X_pool == target_row, axis=1))[0]
        s=0
        if index.size > 0:
            # print(f"Найдена строка {target_row} на индексе {index[0]}")
            if index[0] not in index_list:
                X_study.append(target_row)
                y_study.append(y_pool[index[0]])
                index_list.append(index[0])
                X_pool = np.delete(X_pool, index[0], axis=0)
                y_pool = np.delete(y_pool, index[0])
                print(f"Найдена строка {target_row} на индексе {index[0]}")
            else:
                help_copy1 = target_row.copy()
                help_copy2 = target_row.copy()
                while s == 0:
                    i = 0 
                    step1 = get_steps(help_copy1)
                    step2 = get_steps(help_copy2)
                    print(step1,"step")
                    for i in range(7):
                        help_copy1[i] += step1[i][1]
                        help_copy2[i] += step2[i][-1]
                        index2 = np.where(np.all(np.isclose(X_pool, help_copy1), axis=1))[0]
                        if index2.size > 0 and index2[0] not in index_list:
                            s=1
                            print(f"Найдена1 строка {target_row} на индексе {index2[0]}")
                            X_study.append(help_copy1)
                            y_study.append(y_pool[index2[0]])
                            index_list.append(index2[0]) 
                            X_pool = np.delete(X_pool, index2[0], axis=0)
                            y_pool = np.delete(y_pool, index2[0])
                            break
                        index3 = np.where(np.all(np.isclose(X_pool, help_copy2), axis=1))[0]
                        if index3.size > 0 and index3[0] not in index_list:
                            s=1
                            print(f"Найдена1 строка {target_row} на индексе {index3[0]}")
                            X_study.append(help_copy2)
                            y_study.append(y_pool[index3[0]])
                            index_list.append(index3[0]) 
                            X_pool = np.delete(X_pool, index3[0], axis=0)
                            y_pool = np.delete(y_pool, index3[0])
                            break
                # print(f"Найдена1 строка {target_row} на индексе {index2[0]}")
                # X_study.append(help_copy)
                # y_study.append(y_pool[index2[0]])
                # index_list.append(index2[0]) 
                # X_pool = np.delete(X_pool, index2[0], axis=0)
                # y_pool = np.delete(y_pool, index2[0])
                        
                    
        else:
            help_copy1 = target_row.copy()
            help_copy2 = target_row.copy()
            while s == 0:
                i = 0 
                step1 = get_steps(help_copy1)
                step2 = get_steps(help_copy2)
                print(step1,"step1")
                for i in range(7):
                    help_copy1[i] += step1[i][1]
                    help_copy2[i] += step2[i][-1]
                    index2 = np.where(np.all(np.isclose(X_pool, help_copy1), axis=1))[0]
                    if index2.size > 0 and index2[0] not in index_list:
                        s=1
                        print(f"Найдена1 строка {target_row} на индексе {index2[0]}")
                        X_study.append(help_copy1)
                        y_study.append(y_pool[index2[0]])
                        index_list.append(index2[0]) 
                        X_pool = np.delete(X_pool, index2[0], axis=0)
                        y_pool = np.delete(y_pool, index2[0])
                        break
                    index3 = np.where(np.all(np.isclose(X_pool, help_copy2), axis=1))[0]
                    if index3.size > 0 and index3[0] not in index_list:
                        s=1
                        print(f"Найдена1 строка {target_row} на индексе {index3[0]}")
                        X_study.append(help_copy2)
                        y_study.append(y_pool[index3[0]])
                        index_list.append(index3[0]) 
                        X_pool = np.delete(X_pool, index3[0], axis=0)
                        y_pool = np.delete(y_pool, index3[0])
                        break
            
    return X_study, y_study, index_list

In [4]:
def qbc(committee, X_sample):
    # 1. Преобразуем в тензор с градиентами
    tx = torch.tensor(X_sample, dtype=torch.float32, requires_grad=True)
    
    # 2. Собираем предсказания
    preds = []
    for learner in committee.learner_list:
        model = learner.estimator.model
        model.eval()  # Важно: eval() для новых данных!
        
        with torch.enable_grad():  # Принудительно включаем градиенты
            output = model(tx)
            preds.append(output.mean)
    
    # 3. Вычисляем дисперсию предсказаний
    preds_tensor = torch.stack(preds)  # [n_models, n_samples]
    f_avg = preds_tensor.mean(dim=0)
    loss = torch.var(preds_tensor - f_avg, dim=0).mean()
    
    # 4. Вычисляем градиенты
    tx.grad = None  # Очищаем предыдущие градиенты
    loss.backward()
    
    gradients = tx.grad.detach().numpy() if tx.grad is not None else np.zeros_like(X_sample)
    return loss.item(), gradients

def NA_query_strategy(comittee, X_sample):
    for i in range(1):
        _,grads = qbc(comittee, X_sample) # (N, M) -> N
        grads = [torch.tensor(row, dtype=torch.float32).unsqueeze(0) for row in grads]
        l_grad_bnd = Lbnd(X_sample)
        result = [a - b for a, b in zip(grads, l_grad_bnd)]
        print(result, 'result grads - l_bnd')
        sign = [torch.where(tensor > 0.00001, torch.tensor(1), 
                  torch.where(tensor < 0, torch.tensor(-1), torch.tensor(0))) for tensor in result]
        # sign = [torch.where(tensor > 0, torch.tensor(1)), 
        #   torch.where(tensor < 0, torch.tensor(-1)), torch.where(tensor <= 0.00001, torch.tensor(0)) for tensor in result]
        print(sign, 'sign grads - l_bnd')
        for ind in range(len(sign)):
            real_steps = []
            sign_list = sign[ind].squeeze().tolist()
            steps = get_steps(X_sample[ind])
            x_gen = None
            for index, values in enumerate(sign_list):
                real_steps.append(steps[index][values])
            print(real_steps,'real_steps')
            x_gen = X_sample[ind] + real_steps
            X_sample[ind] = x_gen.copy()
    print(X_sample,"X_sample",i)
    return None, X_sample

In [23]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import modAL

df = pd.read_csv('ourall.csv')
# for i in df.columns:
#     if df[i].dtype is not np.float64:
#         df[i] = df[i].astype(np.float64)
print(df.columns)
df = df[df['B4TOD4'] != 0.005]
df = df[df['OMEGA5'] != 15]
df = df[df['OMEGA5'] != -15]
# scaler = MinMaxScaler()
# Fit and transform the data
# df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
# train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
scaler = MinMaxScaler()
# Fit and transform the data
df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
X_pool = df.drop("PT_LOSS", axis=1).values
y_pool = df["PT_LOSS"].values.reshape(-1, 1)
print(df['B4TOD4'].unique(),'B4TOD4')
print(df['B5TOB4'].unique(),'B5TOB4')
print(df['R45TOB4'].unique(),'R45TOB4')
print(df['OMEGA5'].unique(),'OMEGA5')
print(df['GAMMAU'].unique(),'GAMMAU')
print(df['ALPHA4'].unique(),'ALPHA4')
print(df['RE'].unique(),'RE')

print(len(y_pool))
print(X_pool.shape, y_pool.shape)
# print(X_pool,type(X_pool[100]))
# print(X_pool)
# tX, ty = torch.from_numpy(X_pool).float(), torch.from_numpy(y_pool).float()
# dataset = create_dataset_

Index(['B4TOD4', 'B5TOB4', 'R45TOB4', 'OMEGA5', 'GAMMAU', 'ALPHA4', 'RE',
       'PT_LOSS'],
      dtype='object')
[0.         0.42857143 0.71428571 1.        ] B4TOD4
[0.5 1.  0. ] B5TOB4
[0.        0.3902439 1.       ] R45TOB4
[0.  0.5 1. ] OMEGA5
[0.  1.  0.5] GAMMAU
[0.  0.2 0.4 0.6 0.8 1. ] ALPHA4
[0. 1.] RE
3888
(3888, 7) (3888, 1)


In [25]:
from modAL.models import CommitteeRegressor
kernels = ['matern','rbf','matern','rbf']
nus = [0.5, 0.5, 1.5, 1.5]
# kernels = ['matern','rbf']
# nus = [0.5, 0.5]

np.random.seed(42)
train_idx = np.random.choice(range(X_pool.shape[0]), size=7, replace=False)
X_init= X_pool[train_idx]
y_init = y_pool[train_idx]
X_pool = np.delete(X_pool, train_idx, axis=0)
y_pool = np.delete(y_pool, train_idx)
y_init = np.array(y_init).reshape(-1, 1)
print(X_init,y_init,'data')
# X_init = torch.tensor(X_init, dtype=torch.float32)
# y_init = torch.tensor(y_init, dtype=torch.float32)
# train_idx = np.random.choice(range(X_pool.shape[0]), size=7, replace=False)
# X_test = X_pool[train_idx]

learners = []
for i in range(1):
    gp_model =  GPModelSklearnWrapper(kernel_type=kernels[i], nu=nus[i])
    # gp_model.fit(X_init,y_init)
    al_model = ActiveLearner(estimator=gp_model,
        X_training=X_init,
        y_training=y_init,
                            )
    learners.append(al_model)
    
committee = CommitteeRegressor(
    learner_list=learners,
    query_strategy=NA_query_strategy
)

[[1.         0.         1.         1.         1.         0.
  0.        ]
 [0.71428571 1.         0.         0.5        1.         0.2
  0.        ]
 [0.71428571 1.         0.3902439  0.5        1.         0.4
  1.        ]
 [0.71428571 1.         1.         0.5        0.         1.
  1.        ]
 [0.         0.5        1.         1.         0.5        0.2
  1.        ]
 [1.         0.         0.         1.         0.         0.4
  0.        ]
 [0.         1.         0.         0.         0.5        0.2
  1.        ]] [[0.77272738]
 [0.28293864]
 [0.31927123]
 [0.34847371]
 [0.37459504]
 [0.19392956]
 [0.1956374 ]] data
Iter 1/100 - Loss: 2.017
Iter 11/100 - Loss: 1.584
Iter 21/100 - Loss: 1.129
Iter 31/100 - Loss: 0.703
Iter 41/100 - Loss: 0.436
Iter 51/100 - Loss: 0.415
Iter 61/100 - Loss: 0.408
Iter 71/100 - Loss: 0.388
Iter 81/100 - Loss: 0.381
Iter 91/100 - Loss: 0.365
Iter 100/100 - Loss: 0.348


In [27]:
for i in range(20):
    n_initial = 7
    print(f'__________________________________________{i}_______________________________________')
    # np.random.seed(42)
    train_idx = np.random.choice(range(X_pool.shape[0]), size=n_initial, replace=False)
    X_query = X_pool[train_idx]
    y_query = y_pool[train_idx]
    X_pool = np.delete(X_pool, train_idx, axis=0)
    y_pool = np.delete(y_pool, train_idx)
    _, X_generated = committee.query(X_query)
    x_st, y_st, ind = get_new_y(X_generated[0],X_pool,y_pool)
    print(x_st,y_st)
    for i in range(7):
        y_st[i] = np.array([y_st[i]])
    x_st = np.array(x_st)  # Объединяем по оси 0
    y_st = np.array(y_st)
    print(x_st ,y_st,'x_st y_st')
    tX, ty = torch.from_numpy(x_st).float(), torch.from_numpy(y_st).float()
    print(tX,ty,'tx ty')
    committee.teach(
        tX,
        ty,
        bootstrap=True,
        # update_grid=False
    )


__________________________________________0_______________________________________
[tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]])] result grads - l_bnd
[tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]])] sign grads - l_bnd
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[[0.42857143 0.         1.         0.5        0.5        0.
  0.        ]
 [0.714285

C:\Users\ivan\AppData\Local\Temp\ipykernel_2156\2724789211.py:18: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  loss = torch.var(preds_tensor - f_avg, dim=0).mean()


Iter 21/100 - Loss: 0.642
Iter 31/100 - Loss: 0.174
Iter 41/100 - Loss: -0.214
Iter 51/100 - Loss: -0.409
Iter 61/100 - Loss: -0.513
Iter 71/100 - Loss: -0.662
Iter 81/100 - Loss: -0.783
Iter 91/100 - Loss: -0.902
Iter 100/100 - Loss: -0.968
__________________________________________1_______________________________________


C:\Users\ivan\AppData\Local\Temp\ipykernel_2156\2724789211.py:18: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  loss = torch.var(preds_tensor - f_avg, dim=0).mean()


[tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]])] result grads - l_bnd
[tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]])] sign grads - l_bnd
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[[1.         0.5        1.         1.         0.         0.6
  1.        ]
 [1.         1.         0.         0.5        1.         0.4
  1.        ]
 [0.         0.  

...

Iter 31/100 - Loss: -0.299
Iter 41/100 - Loss: -0.763
Iter 51/100 - Loss: -1.115
Iter 61/100 - Loss: -1.312
Iter 71/100 - Loss: -1.480
Iter 81/100 - Loss: -1.611
Iter 91/100 - Loss: -1.713
Iter 100/100 - Loss: -1.783
__________________________________________18_______________________________________
[tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]])] result grads - l_bnd
[tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]])] sign grads - l_bnd
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_step

C:\Users\ivan\AppData\Local\Temp\ipykernel_2156\2724789211.py:18: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  loss = torch.var(preds_tensor - f_avg, dim=0).mean()


Iter 31/100 - Loss: -0.312
Iter 41/100 - Loss: -0.781
Iter 51/100 - Loss: -1.150
Iter 61/100 - Loss: -1.377
Iter 71/100 - Loss: -1.569
Iter 81/100 - Loss: -1.714
Iter 91/100 - Loss: -1.832
Iter 100/100 - Loss: -1.905
__________________________________________19_______________________________________
[tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]]), tensor([[nan, nan, nan, nan, nan, nan, nan]])] result grads - l_bnd
[tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]]), tensor([[0, 0, 0, 0, 0, 0, 0]])] sign grads - l_bnd
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_steps
[0, 0, 0, 0, 0, 0, 0] real_step

C:\Users\ivan\AppData\Local\Temp\ipykernel_2156\2724789211.py:18: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1831.)
  loss = torch.var(preds_tensor - f_avg, dim=0).mean()


Iter 61/100 - Loss: -1.215
Iter 71/100 - Loss: -1.346
Iter 81/100 - Loss: -1.444
Iter 91/100 - Loss: -1.527
Iter 100/100 - Loss: -1.587


In [213]:
qbc_loss, tx1 = qbc(committee,X_init)

In [214]:
print(qbc_loss,tx1)

0.0008488009916618466 [[-5.15981556e-05  1.00621211e-04 -1.17852280e-04 -3.29608120e-05
   1.12144888e-04  1.53855188e-04 -2.74570302e-05]
 [ 2.32143037e-04  4.15501010e-04 -1.03367853e-03 -4.41779295e-04
   8.29257362e-04  1.65339047e-03  3.84743442e-04]
 [-5.90944546e-05 -1.28400221e-04  3.32137570e-05  9.65578656e-05
   1.70068342e-05 -1.09837099e-04  8.14710365e-05]
 [ 6.46838453e-05 -6.04182569e-05 -1.50804932e-04  5.10180689e-05
  -5.64772745e-06  1.30455563e-04  8.81767264e-05]
 [-3.40892875e-05  6.35359584e-06 -2.68483454e-05  4.79912842e-06
   1.81002833e-05  1.36634662e-05  8.23305891e-06]
 [-8.35812534e-05  1.31484985e-05 -1.27350824e-04  3.99307246e-05
   5.13422856e-05  1.15187628e-04  1.43816578e-05]
 [ 1.22532001e-04 -6.59809302e-06 -3.22147564e-04  8.94698169e-05
   1.02617407e-04  2.38808803e-04  1.54390218e-04]]


In [58]:
grads = tensor = torch.tensor(tx1, dtype=torch.float32)
tensors = []
# for row in grads:
#     # Преобразуем каждую строку в тензор
#     tensor_row = torch.tensor(row, dtype=torch.float32)
#     tensors.append(tensor_row.unsqueeze(0))  # добавляем дополнительную ось, чтобы получился shape [1, 7]
# tensors
result_tensors = [torch.tensor(row, dtype=torch.float32).unsqueeze(0) for row in grads]
result_tensors

C:\Users\ivan\AppData\Local\Temp\ipykernel_12620\2879372577.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  result_tensors = [torch.tensor(row, dtype=torch.float32).unsqueeze(0) for row in grads]


[tensor([[ 7.3109e-05, -1.2254e-04,  2.4533e-04,  9.3342e-05,  1.5056e-04,
          -7.6067e-05, -7.0720e-05]]),
 tensor([[-1.6307e-05,  2.2793e-05, -3.9613e-05, -1.8962e-05, -2.4406e-05,
           1.2174e-05,  1.7706e-05]]),
 tensor([[-1.7619e-05,  2.0817e-05, -3.7381e-05, -1.9497e-05, -2.1484e-05,
           8.1174e-06,  1.0815e-05]]),
 tensor([[-9.0642e-06,  1.1346e-05, -2.4761e-05, -1.0408e-05, -1.0332e-05,
           3.5874e-06,  6.8198e-06]]),
 tensor([[ 1.5760e-05, -2.1192e-05,  3.2812e-05,  1.6572e-05,  1.9343e-05,
          -9.8788e-06, -1.6155e-05]]),
 tensor([[-1.4557e-05,  2.3196e-05, -5.1371e-05, -1.7979e-05, -3.3827e-05,
           1.6267e-05,  1.3891e-05]]),
 tensor([[-1.9446e-05,  5.1341e-05, -9.6295e-05, -3.2109e-05, -5.0338e-05,
           3.1056e-05,  2.1049e-05]])]

In [428]:
a = Lbnd(X_init)
# len(np.unique(committee.X_training, axis=0)) == len(committee.X_training)
print(type(committee.X_training))
print(committee.X_training)

<class 'NoneType'>
None


In [46]:
a

[tensor([[-1.,  1., -1., -1., -1.,  1.,  1.]]),
 tensor([[ 0., -1.,  1.,  0., -1.,  0.,  1.]]),
 tensor([[ 0., -1.,  0.,  0., -1.,  0., -1.]]),
 tensor([[ 0., -1., -1.,  0.,  1., -1., -1.]]),
 tensor([[ 1.,  0., -1., -1.,  0.,  0., -1.]]),
 tensor([[-1.,  1.,  1., -1.,  1.,  0.,  1.]]),
 tensor([[ 1., -1.,  1.,  1.,  0.,  0., -1.]])]

In [31]:
train_idx = np.random.choice(range(X_pool.shape[0]), size=3706, replace=False)
print(len(X_pool))
print(len(train_idx))
X_test = X_pool[train_idx]
y_test = y_pool[train_idx]
print(mean_absolute_error(y_test,committee.predict(X_test)),r2_score(y_test,committee.predict(X_test)))

3741
3706
0.035606299164955404 0.8045895769274313


In [ ]:
0.05962525020862163 0.17514407521478237
